# 📐 Day 22 · Projections: how big is Uzbekistan, really?

**#30DayMapChallenge Uzbekistan 2026** · beginner tutorial, no experience needed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodiraTillayeva/30DayMapChallengeUzbekistan/blob/main/2026/Day%2022%20Projections/tutorial/day22_how_big_is_uzbekistan.ipynb)

The Earth is round, but maps are flat. You can't flatten an orange peel without stretching or tearing it, and you can't flatten the Earth either. Every flat map makes a choice about **what to stretch**. That choice is called a **projection**.

The map you see most often, on Google Maps, Yandex Maps or OpenStreetMap, uses the **Mercator** projection. It keeps shapes nice, but it makes everything far from the equator **look bigger**.

In this notebook you will:
1. draw the same world in two projections,
2. measure how much bigger Uzbekistan looks on a web map,
3. **move Uzbekistan** to the equator, to Scandinavia and to Greenland, and watch it grow,
4. see the stretching with **Tissot circles**,
5. find out which region of Uzbekistan is stretched the most,
6. save a poster, and move any country you like.

Run the cells from top to bottom: click a cell, then press **Shift + Enter**.

## Step 0 · Get ready

We use two files:
- `world_countries.geojson`: the borders of every country, from **Natural Earth** (free to use),
- `uzbekistan_regions.geojson`: the 14 regions of Uzbekistan, the same file as on Day 14.

Both are in **longitude and latitude** (degrees). In geopandas, every layer knows its **CRS** (*coordinate reference system*), and `to_crs` changes it from one projection to another.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, Point

BASE = "https://raw.githubusercontent.com/NodiraTillayeva/30DayMapChallengeUzbekistan/main/2026/Day%2022%20Projections/tutorial/data/"

world = gpd.read_file(BASE + "world_countries.geojson")
regions = gpd.read_file(BASE + "uzbekistan_regions.geojson")
uzbekistan = regions.union_all()

world = world.clip(box(-180, -80, 180, 84))     # Mercator can't show the poles

MERCATOR = "EPSG:3857"       # the web map projection
EQUAL_AREA = "EPSG:6933"     # keeps areas true, for measuring
EQUAL_EARTH = "EPSG:8857"    # keeps areas true, and looks nice

BACK, LAND, LINE = "#0f1720", "#2a3642", "#0f1720"
COLORS = ["#5fa8a8", "#e9c46a", "#f08a4b", "#e63946"]

print(len(world), "countries,", len(regions), "regions of Uzbekistan")
print("CRS:", world.crs)
print("Ready! ✅")

## Step 1 · The same world, two ways

Every projection has a code. **EPSG:4326** is plain longitude and latitude: one degree is one step on the map, everywhere. **EPSG:3857** is **Web Mercator**, the projection of almost every web map.

Same data, same countries. Look at Greenland, Africa and Uzbekistan (in yellow).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BACK)
for ax, crs, name in zip(axes, ["EPSG:4326", MERCATOR], ["longitude and latitude", "Web Mercator"]):
    ax.set_facecolor(BACK)
    world.to_crs(crs).plot(ax=ax, color=LAND, edgecolor=LINE, linewidth=0.3)
    world[world["name"] == "Uzbekistan"].to_crs(crs).plot(ax=ax, color="#e9c46a")
    ax.set_title(name, color="white")
    ax.axis("off")
plt.show()

In Mercator, the lines of latitude get **further apart** as you go north. Greenland looks as big as Africa. In reality, Africa is **14 times bigger**.

## Step 2 · How big does it look, how big is it?

If we measure areas on the Mercator map, we get what the map **shows**. If we measure them in an **equal-area** projection (EPSG:6933), we get the **real** size. Let's do both for a few countries.

In [ ]:
countries = ["Dem. Rep. Congo", "Uzbekistan", "Kazakhstan", "Greenland"]
some = world[world["name"].isin(countries)].set_index("name").loc[countries]

table = pd.DataFrame({
    "looks like, km²": some.to_crs(MERCATOR).area / 1e6,
    "really, km²":     some.to_crs(EQUAL_AREA).area / 1e6,
})
table["looks bigger ×"] = table["looks like, km²"] / table["really, km²"]
table.round({"looks like, km²": -3, "really, km²": -3, "looks bigger ×": 1})

The DR Congo sits on the equator, so Mercator shows it at its true size (×1.0). Uzbekistan, at about 41° north, looks **1.8 times** bigger than it is. Greenland looks **16 times** bigger!

The rule: at latitude φ, Mercator makes areas bigger by 1 / cos²(φ). At 41°, that's 1 / 0.75² ≈ 1.8.

In [ ]:
for lat in [0, 20, 41, 55, 72]:
    print(f"at {lat:2}° north, areas look {1 / np.cos(np.radians(lat))**2:5.1f} times bigger")

## Step 3 · Move Uzbekistan

Now the fun part. We take the **real** outline of Uzbekistan and move it to another place on Earth, keeping its true size. Then we draw it on a Mercator map.

How? `move_to` uses a projection called **LAEA** (*Lambert azimuthal equal-area*) centred on Uzbekistan. In it, Uzbekistan is a flat shape in metres around the point (0, 0). Then we **pretend** those metres are around another point, the new centre, and go back to longitude and latitude. The shape and the size stay the same; only the place changes.

In [ ]:
def move_to(shape, lat, lng):
    c = shape.centroid
    here  = f"+proj=laea +lat_0={c.y} +lon_0={c.x} +datum=WGS84"
    there = f"+proj=laea +lat_0={lat} +lon_0={lng} +datum=WGS84"
    flat = gpd.GeoSeries([shape], crs=4326).to_crs(here)
    return flat.set_crs(there, allow_override=True).to_crs(4326).iloc[0]

spots = pd.DataFrame({
    "place": ["the equator (Congo)", "home", "Scandinavia", "Greenland"],
    "lat":   [0,  41.74, 62, 72],
    "lng":   [22, 63.15, 16, -40],
})
copies = gpd.GeoDataFrame(spots, crs=4326,
    geometry=[move_to(uzbekistan, r.lat, r.lng) for r in spots.itertuples()])

copies["real km²"] = copies.to_crs(EQUAL_AREA).area / 1e6
copies["on the map ×"] = copies.to_crs(MERCATOR).area / copies.to_crs(EQUAL_AREA).area
copies.drop(columns="geometry").round({"real km²": -3, "on the map ×": 1})

All four copies are **the same size** in reality, about 440,000 km². Now let's draw them on the Mercator map.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12), facecolor=BACK)
ax.set_facecolor(BACK)
world.to_crs(MERCATOR).plot(ax=ax, color=LAND, edgecolor=LINE, linewidth=0.3)
copies.to_crs(MERCATOR).plot(ax=ax, color=COLORS)
ax.set_xlim(-7.5e6, 9.5e6)
ax.set_ylim(-4e6, 17e6)
ax.axis("off")
plt.show()

On the equator, Uzbekistan fits inside the Congo. In Scandinavia, it covers Sweden and half of Norway. On Greenland, it looks like a giant.

**Same country, same size. Only the map changed.**

## Step 4 · The fair map: equal area

Now draw the same four copies on **Equal Earth** (EPSG:8857), an **equal-area** projection made in 2018. Here, every country keeps its true size.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7), facecolor=BACK)
ax.set_facecolor(BACK)
world.to_crs(EQUAL_EARTH).plot(ax=ax, color=LAND, edgecolor=LINE, linewidth=0.3)
copies.to_crs(EQUAL_EARTH).plot(ax=ax, color=COLORS)
ax.axis("off")
plt.show()

Now the four copies are the same size. But look at Greenland: the copy there is **squashed sideways**. Equal-area projections keep sizes, but bend shapes. No flat map can keep both.

## Step 5 · Tissot circles

In the 19th century, the French mathematician **Nicolas Auguste Tissot** found a simple way to see how a map stretches: draw **identical circles** all over the Earth and see what the map does to them.

Each of our circles has a radius of 400 km on the real Earth.

In [ ]:
circles = []
for lat in range(-75, 76, 15):
    for lng in range(-165, 180, 30):
        centre = f"+proj=laea +lat_0={lat} +lon_0={lng} +datum=WGS84"
        circle = gpd.GeoSeries([Point(0, 0).buffer(400_000)], crs=centre).to_crs(4326)
        circles.append(circle.iloc[0])
circles = gpd.GeoSeries(circles, crs=4326)

fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor=BACK)
for ax, crs, name in zip(axes, [MERCATOR, EQUAL_EARTH], ["Mercator", "Equal Earth"]):
    ax.set_facecolor(BACK)
    world.to_crs(crs).plot(ax=ax, color=LAND, edgecolor=LINE, linewidth=0.3)
    circles.to_crs(crs).plot(ax=ax, color="#e63946", alpha=0.75)
    ax.set_title(name, color="white")
    ax.axis("off")
plt.show()

- **Mercator**: the circles stay **round** (shapes are kept), but grow **huge** near the poles.
- **Equal Earth**: the circles all have the **same area**, but turn into ovals near the edges.

That's the trade-off of every projection.

## Step 6 · Inside Uzbekistan

Uzbekistan goes from about 37° north (Termez) to 45.6° north (the north of Karakalpakstan). So even inside the country, Mercator stretches some regions more than others.

In [ ]:
regions["real km²"] = regions.to_crs(EQUAL_AREA).area / 1e6
regions["on the map km²"] = regions.to_crs(MERCATOR).area / 1e6
regions["×"] = regions["on the map km²"] / regions["real km²"]

regions.drop(columns="geometry").sort_values("×").round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8), facecolor=BACK)
ax.set_facecolor(BACK)
regions.to_crs(MERCATOR).plot(ax=ax, column="×", cmap="YlOrRd", edgecolor=BACK, linewidth=0.8,
                              legend=True, legend_kwds={"shrink": 0.6, "label": "looks bigger ×"})
bar = fig.axes[-1]
bar.tick_params(colors="white")
bar.yaxis.label.set_color("white")
ax.axis("off")
plt.show()

On a web map, **Karakalpakstan** looks 1.9 times its real size and **Surkhandarya** 1.6 times. So next to each other, Karakalpakstan looks about **18% too big**.

That's why, on Day 21, we switched to a **UTM** projection before measuring buildings: never measure areas on a Mercator map.

## Step 7 · Save your poster

The Mercator map with the four copies, with only their latitudes as tiny tags. It's saved as **how_big_is_uzbekistan.png**.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12), facecolor=BACK)
ax.set_facecolor(BACK)
world.to_crs(MERCATOR).plot(ax=ax, color=LAND, edgecolor=LINE, linewidth=0.3)
merc = copies.to_crs(MERCATOR)
merc.plot(ax=ax, color=COLORS)
for (_, row), color in zip(merc.iterrows(), COLORS):
    x0, y0, x1, y1 = row.geometry.bounds
    ax.text(x1 + 2e5, (y0 + y1) / 2, f"{row['lat']:.0f}°", color=color, fontsize=10,
            family="monospace", va="center")
ax.set_xlim(-7.5e6, 9.5e6)
ax.set_ylim(-4e6, 17e6)
ax.axis("off")
plt.subplots_adjust(0, 0, 1, 1)
plt.savefig("how_big_is_uzbekistan.png", dpi=200, facecolor=BACK)
plt.show()

try:
    from google.colab import files
    files.download("how_big_is_uzbekistan.png")
except ImportError:
    print("Saved how_big_is_uzbekistan.png")

## Step 8 · Move anything, anywhere

`move_to` works for any shape. Move **Greenland** to the equator, or **Uzbekistan** to the far south, near Antarctica. Change the names and the numbers and run the cell again.

In [ ]:
name = "Greenland"
lat, lng = 0, 20

shape = world[world["name"] == name].union_all()
moved = gpd.GeoSeries([move_to(shape, lat, lng)], crs=4326)

fig, ax = plt.subplots(figsize=(12, 10), facecolor=BACK)
ax.set_facecolor(BACK)
world.to_crs(MERCATOR).plot(ax=ax, color=LAND, edgecolor=LINE, linewidth=0.3)
world[world["name"] == name].to_crs(MERCATOR).plot(ax=ax, color="#5b6875")
moved.to_crs(MERCATOR).plot(ax=ax, color="#e63946")
ax.axis("off")
plt.show()

## 🎉 Done!

You projected the world, measured what Mercator does to areas, moved Uzbekistan around the globe, and drew Tissot circles.

Share your map with **#30DayMapChallenge** and **#30DayMapChallengeUzbekistan**, and please credit:
> Borders: Natural Earth. Inspired by thetruesize.com.